# 03 — Modèles, embeddings et contrôles de chargement

**Objectif** : distinguer baseline locale, vrai modèle open-weight, Ollama et fournisseur online sans introduire de téléchargement ou d'appel externe caché.

**Critère de passage** : la baseline fonctionne sans clé ; tout autre modèle est explicitement disponible, localisé et mesuré.

In [1]:
from pathlib import Path
import os
import sys

root_hint = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root_hint / 'notebooks'))
from helpers import bootstrap, display_table, module_available, safe_model_summary

ROOT = bootstrap()
safe_model_summary()

{'baseline': 'hash embeddings déterministes 384 dimensions',
 'local_embedding_model': 'intfloat/multilingual-e5-small',
 'local_model_path_exists': False,
 'sentence_transformers_installed': False,
 'gemini_configured': True,
 'gemini_sdk_installed': True,
 'ollama_url': 'http://127.0.0.1:11434'}

In [2]:
from app.retrieval.dense import _hash_embedding

texts = [
    'capitaux propres du groupe en 2025',
    'montant des capitaux propres publiés',
    'satisfaction des utilisateurs MyFoyer',
]
vectors = [_hash_embedding(text) for text in texts]
similarity = lambda a, b: sum(x * y for x, y in zip(a, b, strict=True))
rows = [
    {'a': texts[0], 'b': text, 'cosine': round(similarity(vectors[0], vector), 4)}
    for text, vector in zip(texts, vectors, strict=True)
]
assert len(vectors[0]) == 384
display_table(rows)

,a,b,cosine
0,capitaux propres du groupe en 2025,capitaux propres du groupe en 2025,1.0000
1,capitaux propres du groupe en 2025,montant des capitaux propres publiés,0.5336
2,capitaux propres du groupe en 2025,satisfaction des utilisateurs MyFoyer,-0.0928


In [3]:
local_path = os.getenv('LOCAL_EMBEDDING_MODEL_PATH', '').strip()
local_model = None
if local_path and module_available('sentence_transformers'):
    model_path = Path(local_path).expanduser()
    if model_path.exists():
        from sentence_transformers import SentenceTransformer
        local_model = SentenceTransformer(str(model_path), local_files_only=True)
        embeddings = local_model.encode(texts, normalize_embeddings=True)
        print({'model_path': str(model_path), 'shape': tuple(embeddings.shape)})
    else:
        print('Le chemin du modèle local est configuré mais inexistant.')
else:
    print('Aucun modèle open-weight local chargé : la baseline reste active.')

Aucun modèle open-weight local chargé : la baseline reste active.


In [4]:
from urllib.error import URLError
from urllib.request import urlopen

ollama_url = os.getenv('OLLAMA_BASE_URL', 'http://127.0.0.1:11434').rstrip('/')
if ollama_url.startswith(('http://127.0.0.1:', 'http://localhost:')):
    try:
        with urlopen(f'{ollama_url}/api/tags', timeout=0.5) as response:
            print({'ollama_available': True, 'status': response.status})
    except (URLError, TimeoutError, OSError):
        print({'ollama_available': False, 'message': 'Aucun service Ollama local détecté.'})
else:
    print('URL Ollama non locale : test désactivé par précaution.')

{'ollama_available': True, 'status': 200}


In [5]:
RUN_ONLINE_EXPERIMENT = os.getenv('RUN_ONLINE_EXPERIMENT', '0') == '1'
gemini_ready = bool(os.getenv('GEMINI_API_KEY')) and module_available('google.genai')
if RUN_ONLINE_EXPERIMENT and gemini_ready:
    from google import genai
    client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
    embedding_model = os.environ['GEMINI_EMBEDDING_MODEL']
    generation_model = os.environ['GEMINI_GENERATION_MODEL']
    embedding_response = client.models.embed_content(
        model=embedding_model, contents=texts,
    )
    generation_response = client.models.generate_content(
        model=generation_model,
        contents='Réponds uniquement par OK : test de connectivité contrôlé.',
    )
    print({
        'embedding_model': embedding_model,
        'embedding_vectors': len(embedding_response.embeddings),
        'embedding_dimensions': len(embedding_response.embeddings[0].values),
        'generation_model': generation_model,
        'generation_response_received': bool(generation_response.text),
    })
else:
    print({'gemini_ready': gemini_ready, 'online_calls_enabled': RUN_ONLINE_EXPERIMENT})

{'gemini_ready': True, 'online_calls_enabled': False}


### Décision d'architecture

La baseline locale est un outil de validation du pipeline, pas un substitut équivalent à un modèle d'embedding entraîné. Un modèle open-weight doit être chargé depuis des poids déjà disponibles ; un fournisseur online exige un choix explicite et une revue des données envoyées.